In [89]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [90]:
words = open('names.txt','r').read().splitlines()

In [91]:
chars = sorted(list(set(''.join(words))))

In [93]:
# instead of the predicting the next letter wrt to the current letter we take the context of more than one letter 
#this is the block size
#X,Y is the dataset where X has the context and Y has the output letter
block_size = 3
X , Y = [] , []
for word in words:
    context = [0]*block_size
    for ch in word + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]
X = torch.tensor(X)
Y = torch.tensor(Y)


In [109]:
# need to embed the charecters into a vector of some dimension here 2
# randomly generate here and using the loss function and the backpropagation we can find the best embeddings
num_dims = 2
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,num_dims),generator = g)
embd = C[X]
W1 = torch.randn((6,100),generator = g)
b1 = torch.randn(100,generator = g)
W2 = torch.randn((100,27),generator = g)
b2 = torch.randn((27),generator = g)
#params
params = [C,W1,b1,W2,b2]
params
for p in params:
    p.requires_grad = True

In [ ]:
#training
for i in range(100):
    #forward pass
    embd = C[X]
    l1 = torch.tanh(embd.view(embd.shape[0],block_size * num_dims) @ W1 + b1)
    l2 = l1 @ W2 + b2
    loss = F.cross_entropy(l2,Y)
    print(loss.item())
    
    #backward pass
    for p in params:
        p.grad = None
    loss.backward()

    #update
    for p in params:
        p.data -= 0.1*p.grad
    

In [126]:
#instead of calculating the loss on all of them we can use a mini batch for training
for i in range(10000):
    #forward pass
    ix = torch.randint(0,X.shape[0],(32,))
    embd = C[X[ix]]
    l1 = torch.tanh(embd.view(embd.shape[0],block_size * num_dims) @ W1 + b1)
    l2 = l1 @ W2 + b2
    loss = F.cross_entropy(l2,Y[ix])
    # print(loss.item())
    
    #backward pass
    for p in params:
        p.grad = None
    loss.backward()

    #update
    for p in params:
        p.data -= 0.01*p.grad
print(loss.item())

2.3955910205841064


In [127]:
#total loss
embd = C[X]
l1 = torch.tanh(embd.view(embd.shape[0],block_size * num_dims) @ W1 + b1)
l2 = l1 @ W2 + b2
loss = F.cross_entropy(l2,Y)
loss

tensor(2.3021, grad_fn=<NllLossBackward0>)